# Notebook 01: Data Acquisition

## HIV Drug Resistance Prediction with ESM-2

---

**Objective**: Download and parse Stanford HIVDB sequence and phenotype data.

**Data Source**: Stanford HIV Drug Resistance Database (HIVDB)
- https://hivdb.stanford.edu/

**Expected Output**:
- 6,308 sequences (2,171 PI, 1,867 NRTI, 2,270 NNRTI)
- Resistance annotations for 18 drugs

---

In [ ]:
# Install dependencies
# !pip install -q pandas numpy biopython tqdm

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from data_processing import (
    get_drug_list, load_fasta, save_fasta, 
    parse_hivdb_sequences, get_dataset_statistics
)

print("Imports complete")

In [ ]:
# Project paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
DATA_RAW = DATA_DIR / 'raw'
DATA_PROCESSED = DATA_DIR / 'processed'

# Create directories
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")

## 1. Download HIVDB Data

Stanford HIVDB provides genotype-phenotype datasets for download.

**Manual download instructions:**
1. Visit https://hivdb.stanford.edu/pages/genopheno.dataset.html
2. Download the genotype-phenotype datasets for:
   - Protease Inhibitors (PI)
   - NRTIs
   - NNRTIs
3. Place files in `data/raw/`

In [ ]:
# Check for downloaded data
expected_files = [
    'PI_DataSet.txt',
    'NRTI_DataSet.txt', 
    'NNRTI_DataSet.txt'
]

print("Checking for HIVDB data files...")
for filename in expected_files:
    filepath = DATA_RAW / filename
    if filepath.exists():
        size_mb = filepath.stat().st_size / 1e6
        print(f"  Found: {filename} ({size_mb:.2f} MB)")
    else:
        print(f"  Missing: {filename}")
        print(f"    Download from https://hivdb.stanford.edu/pages/genopheno.dataset.html")

## 2. Parse HIVDB Data

Extract sequences and phenotype labels from HIVDB format.

In [ ]:
def parse_hivdb_dataset(filepath):
    """
    Parse Stanford HIVDB genotype-phenotype dataset.
    
    The format is a tab-separated file with:
    - Sequence information
    - Drug resistance fold-change values
    - Resistance classifications
    """
    df = pd.read_csv(filepath, sep='\t', low_memory=False)
    print(f"  Loaded {len(df)} records")
    print(f"  Columns: {df.columns.tolist()[:10]}...")
    return df

In [ ]:
# Parse each drug class
datasets = {}

for drug_class in ['PI', 'NRTI', 'NNRTI']:
    filepath = DATA_RAW / f'{drug_class}_DataSet.txt'
    
    if filepath.exists():
        print(f"\nParsing {drug_class}...")
        datasets[drug_class] = parse_hivdb_dataset(filepath)
    else:
        print(f"\nSkipping {drug_class} (file not found)")

## 3. Extract Sequences and Phenotypes

Separate sequence data from resistance phenotypes.

In [ ]:
def extract_sequences_and_phenotypes(df, drug_class):
    """
    Extract sequences and phenotype labels from HIVDB dataset.
    """
    # Common sequence column names in HIVDB
    seq_cols = ['SeqID', 'IsolateID', 'Subtype', 'AASequence', 'NASequence']
    
    # Identify sequence column
    aa_col = None
    for col in df.columns:
        if 'AA' in col.upper() and 'SEQ' in col.upper():
            aa_col = col
            break
        elif 'SEQUENCE' in col.upper():
            aa_col = col
    
    if aa_col is None:
        print(f"  Warning: Could not find sequence column")
        return None, None
    
    # Get drug-specific columns (fold-change and classification)
    drugs = get_drug_list(drug_class)
    
    phenotype_cols = []
    for drug in drugs:
        for suffix in ['_FC', '_class3', '_class2']:
            col = f"{drug}{suffix}"
            if col in df.columns:
                phenotype_cols.append(col)
    
    # Extract data
    sequences = df[aa_col].tolist()
    seq_ids = df['SeqID'].tolist() if 'SeqID' in df.columns else [f"seq_{i}" for i in range(len(df))]
    
    phenotypes = df[phenotype_cols].copy() if phenotype_cols else pd.DataFrame()
    phenotypes['seq_id'] = seq_ids
    
    print(f"  Extracted {len(sequences)} sequences")
    print(f"  Phenotype columns: {len(phenotype_cols)}")
    
    return sequences, seq_ids, phenotypes

In [ ]:
# Extract sequences and phenotypes for each drug class
processed_data = {}

for drug_class, df in datasets.items():
    print(f"\nProcessing {drug_class}...")
    sequences, seq_ids, phenotypes = extract_sequences_and_phenotypes(df, drug_class)
    
    if sequences:
        # Filter sequences (remove those with gaps or unknown amino acids)
        valid_mask = [all(c.isalpha() and c != 'X' for c in seq.replace('-', '')) 
                     for seq in sequences]
        
        valid_sequences = [s for s, v in zip(sequences, valid_mask) if v]
        valid_seq_ids = [s for s, v in zip(seq_ids, valid_mask) if v]
        valid_phenotypes = phenotypes[valid_mask].reset_index(drop=True)
        
        processed_data[drug_class] = {
            'sequences': valid_sequences,
            'seq_ids': valid_seq_ids,
            'phenotypes': valid_phenotypes
        }
        
        print(f"  After filtering: {len(valid_sequences)} sequences")

## 4. Dataset Statistics

Compute summary statistics for the processed data.

In [ ]:
print("\n" + "="*60)
print("DATASET STATISTICS")
print("="*60)

total_seqs = 0

for drug_class, data in processed_data.items():
    n_seqs = len(data['sequences'])
    total_seqs += n_seqs
    
    seq_lengths = [len(s) for s in data['sequences']]
    
    print(f"\n{drug_class}:")
    print(f"  Sequences: {n_seqs:,}")
    print(f"  Sequence length: {np.mean(seq_lengths):.0f} +/- {np.std(seq_lengths):.0f} aa")
    print(f"  Range: {np.min(seq_lengths)}-{np.max(seq_lengths)} aa")
    
    # Drug-specific statistics
    drugs = get_drug_list(drug_class)
    print(f"  Drugs ({len(drugs)}): {drugs}")

print(f"\nTotal sequences: {total_seqs:,}")

In [ ]:
# Resistance prevalence by drug
print("\nResistance prevalence by drug:")
print("-" * 50)

for drug_class, data in processed_data.items():
    print(f"\n{drug_class}:")
    
    phenotypes = data['phenotypes']
    drugs = get_drug_list(drug_class)
    
    for drug in drugs:
        col = f"{drug}_class2"
        if col in phenotypes.columns:
            labels = phenotypes[col].dropna()
            if len(labels) > 0:
                n_resistant = labels.sum()
                prevalence = 100 * n_resistant / len(labels)
                print(f"  {drug}: {n_resistant:>4}/{len(labels):<4} ({prevalence:5.1f}% resistant)")

## 5. Save Processed Data

Save sequences and phenotypes for downstream analysis.

In [ ]:
print("\nSaving processed data...")

for drug_class, data in processed_data.items():
    # Save sequences as FASTA
    fasta_path = DATA_PROCESSED / f'{drug_class}_sequences.fasta'
    save_fasta(data['sequences'], data['seq_ids'], fasta_path)
    print(f"  {fasta_path.name}")
    
    # Save phenotypes as CSV
    pheno_path = DATA_PROCESSED / f'{drug_class}_phenotypes.csv'
    data['phenotypes'].to_csv(pheno_path, index=False)
    print(f"  {pheno_path.name}")

# Save metadata
import json

metadata = {
    'date_processed': pd.Timestamp.now().isoformat(),
    'drug_classes': list(processed_data.keys()),
    'counts': {dc: len(data['sequences']) for dc, data in processed_data.items()},
    'total_sequences': total_seqs
}

with open(DATA_PROCESSED / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("\nDone!")

## Summary

**Expected results:**
- PI: 2,171 sequences, 8 drugs
- NRTI: 1,867 sequences, 6 drugs
- NNRTI: 2,270 sequences, 4 drugs
- Total: 6,308 sequences, 18 drugs

**Next steps:**
- Notebook 02: Develop baseline model using binary mutation encoding
- Notebook 03: Extract ESM-2 embeddings